In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

from DataPreprocessing import smart_imputer, smart_jitter

In [2]:
path = "/Users/zuzannasosnowska/.cache/kagglehub/datasets/krzysztofjamroz/apartment-prices-in-poland/versions/9/apartments_pl_2024_06.csv"

df = pd.read_csv(path)

df["price_per_m2"] = df["price"] / df["squareMeters"]

polish_cities_names = {'szczecin': 'Szczecin', 'gdynia': 'Gdynia', 'krakow': 'Kraków',  'poznan': 'Poznań', 'bialystok': 'Białystok',  'gdansk': 'Gdańsk', 'wroclaw': 'Wrocław', 'radom': 'Radom', 'rzeszow': 'Rzeszów', 'lodz': 'Łódź', 'katowice': 'Katowice', 'lublin': 'Lublin', 'czestochowa': 'Częstochowa', 'warszawa': 'Warszawa', 'bydgoszcz': 'Bydgoszcz'}

df['city'] = [polish_cities_names[city] for city in df['city']]

has_columns = ['hasParkingSpace', 'hasBalcony', 'hasElevator', 'hasSecurity', 'hasStorageRoom']
for column in has_columns:
    df[column] = df[column].eq('yes').mul(1)

df = df.drop(columns=['id'], axis=1)

In [8]:
print("Liczba duplikatów (rekordy z tych samych ogłoszeń): ", df.duplicated().sum())
df = df.drop_duplicates()
print("Liczba duplikatów po usunięciu (rekordy z tych samych ogłoszeń): ",df.duplicated().sum())

Liczba duplikatów (rekordy z tych samych ogłoszeń):  208
Liczba duplikatów po usunięciu (rekordy z tych samych ogłoszeń):  0


In [9]:
print("Liczba duplikatów geograficznych (nieruchomości w tych samych lokalizacjach): ", df.duplicated(subset=['latitude', 'longitude']).sum())

Liczba duplikatów geograficznych (nieruchomości w tych samych lokalizacjach):  5431


In [10]:
first_match_columns = ['city', 'type', 'rooms', 'floor', 'floorCount', 'buildYear', 'centreDistance', 'poiCount', 'schoolDistance', 'clinicDistance', 'postOfficeDistance', 'kindergartenDistance', 'restaurantDistance', 'collegeDistance', 'pharmacyDistance', 'ownership', 'buildingMaterial', 'condition', 'hasParkingSpace', 'hasBalcony', 'hasElevator', 'hasSecurity', 'hasStorageRoom']

mean_match_columns = ['squareMeters', 'price', 'price_per_m2']

df_no_coords_repeated = (
    df
    .groupby(['latitude', 'longitude'])
    .agg({
        **{column: 'first' for column in first_match_columns},
        **{column: 'mean' for column in mean_match_columns}
    })
    .reset_index()
)

In [11]:
print("Liczba duplikatów geograficznych po agregacji: ", df_no_coords_repeated.duplicated(subset=['latitude', 'longitude']).sum())

Liczba duplikatów geograficznych po agregacji:  0


In [12]:
city_dataframes = {city: df[df["city"] == city] for city in df.city.unique()}
cities = df["city"].unique()

In [14]:
df, info = smart_imputer(df)
df_no_coords_repeated = smart_imputer(df_no_coords_repeated)[0]

/Users/zuzannasosnowska/PycharmProjects/UniversityProjects/Semestr 7/Praca_inżynierska/functions.py:138: UserWarning: Kolumna "buildingMaterial" (kategoryczna) ma >40% braków → pomijam.
  warnings.warn(f'Kolumna "{column}" (kategoryczna) ma >40% braków → pomijam.')
/Users/zuzannasosnowska/PycharmProjects/UniversityProjects/Semestr 7/Praca_inżynierska/functions.py:138: UserWarning: Kolumna "condition" (kategoryczna) ma >40% braków → pomijam.
  warnings.warn(f'Kolumna "{column}" (kategoryczna) ma >40% braków → pomijam.')
/Users/zuzannasosnowska/PycharmProjects/UniversityProjects/Semestr 7/Praca_inżynierska/functions.py:138: UserWarning: Kolumna "condition" (kategoryczna) ma >40% braków → pomijam.
  warnings.warn(f'Kolumna "{column}" (kategoryczna) ma >40% braków → pomijam.')


In [15]:
print(info)

                  column         type  missing_ratio   method
0           squareMeters      numeric       0.000000     none
1                  rooms      numeric       0.000000     none
2                  floor      numeric       0.166299   median
3             floorCount      numeric       0.009675     drop
4              buildYear      numeric       0.157094   median
5               latitude      numeric       0.000000     none
6              longitude      numeric       0.000000     none
7         centreDistance      numeric       0.000000     none
8               poiCount      numeric       0.000000     none
9         schoolDistance      numeric       0.000517     drop
10        clinicDistance      numeric       0.002865     drop
11    postOfficeDistance      numeric       0.000939     drop
12  kindergartenDistance      numeric       0.000892     drop
13    restaurantDistance      numeric       0.001456     drop
14       collegeDistance      numeric       0.026957     drop
15      

In [16]:
df = pd.get_dummies(
    df,
    columns=["city", "type", "ownership", "buildingMaterial", "condition"],
    dtype=int
)

df_no_coords_repeated = pd.get_dummies(
    df_no_coords_repeated,
    columns=["city", "type", "ownership", "buildingMaterial", "condition"],
    dtype=int
)

df

,squareMeters,rooms,floor,floorCount,buildYear,latitude,longitude,centreDistance,poiCount,schoolDistance,...,type_apartmentBuilding,type_blockOfFlats,type_tenement,ownership_condominium,ownership_cooperative,ownership_udział,buildingMaterial_brick,buildingMaterial_concreteSlab,condition_low,condition_premium
0,47.00,2.0,6.0,12.0,1981.0,53.428544,14.552812,0.79,67.0,0.288,...,0,1,0,1,0,0,0,1,0,0
1,88.22,3.0,1.0,2.0,2000.0,53.449093,14.516844,4.09,0.0,0.509,...,1,0,0,1,0,0,1,0,0,1
2,117.00,5.0,4.0,4.0,1993.0,53.443096,14.561348,2.19,10.0,0.216,...,1,0,0,0,0,1,1,0,0,1
3,33.31,1.0,1.0,4.0,1963.0,53.436100,14.541200,1.93,39.0,0.073,...,0,1,0,0,1,0,1,0,0,0
4,56.00,3.0,7.0,7.0,2018.0,53.447465,14.557811,2.68,10.0,0.268,...,0,1,0,1,0,0,1,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21496,54.00,3.0,4.0,4.0,1993.0,53.126520,18.045220,3.01,16.0,0.325,...,0,1,0,1,0,0,0,0,0,0
21497,98.00,3.0,3.0,2.0,1925.0,53.124655,18.008459,0.59,49.0,0.113,...,0,0,1,1,0,0,1,0,0,0
21498,108.96,5.0,2.0,4.0,1889.0,53.131748,18.000648,1.08,28.0,0.281,...,0,0,1,1,0,0,1,0,0,0
21499,50.12,2.0,1.0,1.0,1993.0,53.129657,18.003888,0.88,42.0,0.174,...,0,1,0,1,0,0,1,0,0,0


## Model 1: Regresja liniowa

In [17]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import KFold, cross_val_score
from sklearn.decomposition import PCA
from sklearn.feature_selection import SequentialFeatureSelector, RFECV

In [18]:
y1 = df["price"]
y2 = df["price_per_m2"]
X = df.drop(columns=["price", "price_per_m2"], axis=1)
X1 = df[["latitude", "longitude", "squareMeters"]]
X2 = df[["latitude", "longitude", "rooms", "floorCount", "buildYear", "poiCount", "clinicDistance", "restaurantDistance"]]

X_train, X_test, y_train, y_test = train_test_split(X, y1, test_size=0.2, random_state=42)
X_train2, X_test2, y_train2, y_test2 = train_test_split(X, y2, test_size=0.2, random_state=42)
X2_train, X2_test, y2_train, y2_test = train_test_split(X1, y1, test_size=0.2, random_state=42)
X2_train2, X2_test2, y2_train2, y2_test2 = train_test_split(X2, y2, test_size=0.2, random_state=42)

#### Regresja - modelowanie całkowitej ceny nieruchomości dla pełnego modelu (wszystkie miasta i kolumny)

In [21]:
model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

result_df = pd.DataFrame({"R2_score": [r2_score(y_test, y_pred)],
              "MAE": [mean_absolute_error(y_test, y_pred)],
              "MSE": [mean_squared_error(y_test, y_pred)],
              "RMSE": [root_mean_squared_error(y_test, y_pred)]},
             index=["price"]).round({'R2_score': 3, 'MAE': 2, 'MSE': 2, 'RMSE': 2})
print(result_df.to_latex(formatters={'R2_score': '{:.3f}'.format, 'MAE': '{:.2f}'.format, 'MSE': '{:.2f}'.format, 'RMSE': '{:.2f}'.format}))
result_df

\begin{tabular}{lrrrr}
\toprule
 & R2_score & MAE & MSE & RMSE \\
\midrule
price & 0.785 & 140585.21 & 39965080511.95 & 199912.68 \\
\bottomrule
\end{tabular}



,R2_score,MAE,MSE,RMSE
price,0.785,140585.21,3.996508e+10,199912.68


In [22]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(model, X, y1, cv=kf, scoring='r2')
result_df = pd.DataFrame({"Mean R2-score": scores.mean(), "Std": scores.std()}, index=["price"])
print(result_df.to_latex())
result_df

\begin{tabular}{lrr}
\toprule
 & Mean R2-score & Std \\
\midrule
price & 0.784336 & 0.006632 \\
\bottomrule
\end{tabular}



,Mean R2-score,Std
price,0.784336,0.006632


In [ ]:
coeffs = [model.intercept_] + list(model.coef_)

pd.DataFrame({"Współczynniki": [coeff for coeff in coeffs]}, index=["a_0"] + list(X.columns))

#### Regresja - modelowanie ceny za $m^2$ nieruchomości dla pełnego modelu (wszystkie miasta i kolumny)

In [23]:
model = LinearRegression()
model.fit(X_train2, y_train2)

y_pred2 = model.predict(X_test2)

result_df = pd.DataFrame(
    {"R2_score": [r2_score(y_test2, y_pred2)],
     "MAE": [mean_absolute_error(y_test2, y_pred2)],
     "MSE": [mean_squared_error(y_test2, y_pred2)],
     "RMSE": [root_mean_squared_error(y_test2, y_pred2)]},
    index=["price per m^2"])

print(result_df.to_latex(formatters={'R2_score': '{:.3f}'.format, 'MAE': '{:.2f}'.format, 'MSE': '{:.2f}'.format, 'RMSE': '{:.2f}'.format}))
result_df

\begin{tabular}{lrrrr}
\toprule
 & R2_score & MAE & MSE & RMSE \\
\midrule
price per m^2 & 0.729 & 2067.32 & 7512673.75 & 2740.93 \\
\bottomrule
\end{tabular}



,R2_score,MAE,MSE,RMSE
price per m^2,0.729044,2067.320947,7.512674e+06,2740.92571


In [24]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(model, X, y2, cv=kf, scoring='r2')
pd.DataFrame({"Mean R2-score": scores.mean(), "Std": scores.std()}, index=["price per m^2"])

,Mean R2-score,Std
price per m^2,0.728825,0.007961


#### Regresja - modelowanie całkowitej ceny nieruchomości dla niepełnego modelu (wszystkie miasta)

Kolumny brane pod uwagę:
- `squareMeters` - metraż mieszkania
- `latitude` i `longitude` - współrzędne geograficzne nieruchomości

In [25]:
model = LinearRegression()
model.fit(X2_train, y2_train)

y2_pred = model.predict(X2_test)

pd.DataFrame({"R2_score": [r2_score(y2_test, y2_pred)],
              "MAE": [mean_absolute_error(y2_test, y2_pred)],
              "MSE": [mean_squared_error(y2_test, y2_pred)],
              "RMSE": [root_mean_squared_error(y2_test, y2_pred)]},
             index=["price"])

,R2_score,MAE,MSE,RMSE
price,0.51691,218955.644623,8.959716e+10,299327.85317


In [26]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(model, X1, y1, cv=kf, scoring='r2')
pd.DataFrame({"Mean R2-score": scores.mean(), "Std": scores.std()}, index=["price"])

,Mean R2-score,Std
price,0.513353,0.010751


#### Regresja - modelowanie ceny nieruchomości za $m^2$ dla niepełnego modelu (wszystkie miasta)

Kolumny brane pod uwagę:
- `latitude` i `longitude` - współrzędne geograficzne mieszkania
- `rooms` - liczba pokoi w mieszkaniu
- `floorCount` - ilość pięter w budynku
- `buildYear` - rok budowy mieszkania
- `poiCount` - liczba miejsc publicznych w promieniu 500 m od nieruchomości
- `clinicDistance` - odległość od szpitala
- `restaurantDistance` - odległość od restauracji

In [27]:
model = LinearRegression()
model.fit(X2_train2, y2_train2)

y2_pred2 = model.predict(X2_test2)

pd.DataFrame({"R2_score": [r2_score(y2_test2, y2_pred2)],
              "MAE": [mean_absolute_error(y2_test2, y2_pred2)],
              "MSE": [mean_squared_error(y2_test2, y2_pred2)],
              "RMSE": [root_mean_squared_error(y2_test2, y2_pred2)]},
             index=["price per m^2"])

,R2_score,MAE,MSE,RMSE
price per m^2,0.259816,3526.060521,2.052276e+07,4530.205572


In [28]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(model, X2, y2, cv=kf, scoring='r2')
pd.DataFrame({"Mean R2-score": scores.mean(), "Std": scores.std()}, index=["price per m^2"])

,Mean R2-score,Std
price per m^2,0.257412,0.01119


### Redukcja składowych z użyciem `SequentialFeatureSelector` i redukcja wymiarowości z użyciem `PCA`

In [29]:
pca = PCA()

scores_mean = []
scores_std = []
for i in range(2, X.shape[1], 3):
    pca = PCA(n_components=i)
    transformed_X = pca.fit_transform(X)

    scores = cross_val_score(model, transformed_X, y1, cv=kf, scoring='r2')
    scores_mean.append(np.mean(scores))
    scores_std.append(np.std(scores))
pd.DataFrame({"N-dimensions": range(2, X.shape[1], 3), "Mean R2-score": scores_mean, "Std": scores_std}, index=range(2, X.shape[1], 3))

,N-dimensions,Mean R2-score,Std
2,2,0.285671,0.011009
5,5,0.541542,0.012919
8,8,0.574365,0.012674
11,11,0.626320,0.009681
14,14,0.629206,0.010016
17,17,0.639743,0.009442
20,20,0.653568,0.010897
23,23,0.682816,0.010651
26,26,0.718072,0.008941
29,29,0.745615,0.007984


In [24]:
model = LinearRegression()

r2_scores = []
cols = df.drop(columns=["price", "price_per_m2"], axis=1).columns
columns_with_features = []
scaler = StandardScaler()
for i in range(1, X.shape[1]):
    sfs = SequentialFeatureSelector(model, n_features_to_select=i)
    X_scaled = scaler.fit_transform(X)
    sfs.fit(X_scaled, y1)
    X_sfs = sfs.transform(X_scaled)
    model.fit(X_sfs, y1)

    r2_scores.append(model.score(X_sfs, y1))
    columns_with_features.append(sfs.get_feature_names_out(input_features=cols))

In [25]:
new_added_col = [columns_with_features[0][0]]
for i in range(len(columns_with_features)-1):
    wynik = np.setdiff1d(columns_with_features[i+1], columns_with_features[i])
    new_added_col.append(wynik[0])

result_df = pd.DataFrame({"R2-scores": r2_scores, "Added column": new_added_col, "Number of columns in model": range(1, len(columns_with_features)+1)})

print(result_df.to_latex(caption="Regresja liniowa - Sequential Feature Selector dla ceny całkowitej"))
result_df

\begin{table}
\caption{Regresja liniowa - Sequential Feature Selector dla ceny całkowitej}
\begin{tabular}{lrlr}
\toprule
 & R2-scores & Added column & Number of columns in model \\
\midrule
0 & 0.450627 & squareMeters & 1 \\
1 & 0.580015 & city_Warszawa & 2 \\
2 & 0.643702 & city_Kraków & 3 \\
3 & 0.677506 & type_apartmentBuilding & 4 \\
4 & 0.693627 & latitude & 5 \\
5 & 0.717570 & city_Wrocław & 6 \\
6 & 0.725931 & city_Bydgoszcz & 7 \\
7 & 0.742187 & poiCount & 8 \\
8 & 0.748814 & hasElevator & 9 \\
9 & 0.757054 & city_Szczecin & 10 \\
10 & 0.758561 & city_Gdańsk & 11 \\
11 & 0.762011 & condition_low & 12 \\
12 & 0.762836 & buildYear & 13 \\
13 & 0.763985 & city_Rzeszów & 14 \\
14 & 0.764605 & city_Katowice & 15 \\
15 & 0.765639 & city_Lublin & 16 \\
16 & 0.766568 & condition_premium & 17 \\
17 & 0.766654 & city_Częstochowa & 18 \\
18 & 0.767138 & ownership_cooperative & 19 \\
19 & 0.767139 & kindergartenDistance & 20 \\
20 & 0.767293 & city_Radom & 21 \\
21 & 0.769148 & rooms & 22

,R2-scores,Added column,Number of columns in model
0,0.450627,squareMeters,1
1,0.580015,city_Warszawa,2
2,0.643702,city_Kraków,3
3,0.677506,type_apartmentBuilding,4
4,0.693627,latitude,5
5,0.717570,city_Wrocław,6
6,0.725931,city_Bydgoszcz,7
7,0.742187,poiCount,8
8,0.748814,hasElevator,9
9,0.757054,city_Szczecin,10


# Model GWR

In [32]:
from mgwr.gwr import GWR, MGWR
from mgwr.sel_bw import Sel_BW

import geopandas as gpd
from Models import SklearnGWR

In [34]:
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(x=df['longitude'], y=df['latitude']),
    crs="EPSG:4326"     # Dane stopnie geograficzne
)

gdf = gdf.to_crs("EPSG:2180")   # Transformacja do układu w metrach

gdf

,squareMeters,rooms,floor,floorCount,buildYear,latitude,longitude,centreDistance,poiCount,schoolDistance,...,type_blockOfFlats,type_tenement,ownership_condominium,ownership_cooperative,ownership_udział,buildingMaterial_brick,buildingMaterial_concreteSlab,condition_low,condition_premium,geometry
0,47.00,2.0,6.0,12.0,1981.0,53.428544,14.552812,0.79,67.0,0.288,...,1,0,1,0,0,0,1,0,0,POINT (204685.778 627379.842)
1,88.22,3.0,1.0,2.0,2000.0,53.449093,14.516844,4.09,0.0,0.509,...,0,0,1,0,0,1,0,0,1,POINT (202442.549 629812.934)
2,117.00,5.0,4.0,4.0,1993.0,53.443096,14.561348,2.19,10.0,0.216,...,0,0,0,0,1,1,0,0,1,POINT (205353.087 628961.497)
3,33.31,1.0,1.0,4.0,1963.0,53.436100,14.541200,1.93,39.0,0.073,...,1,0,0,1,0,1,0,0,0,POINT (203967.751 628267.673)
4,56.00,3.0,7.0,7.0,2018.0,53.447465,14.557811,2.68,10.0,0.268,...,1,0,1,0,0,1,0,0,1,POINT (205148.789 629461.632)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21496,54.00,3.0,4.0,4.0,1993.0,53.126520,18.045220,3.01,16.0,0.325,...,1,0,1,0,0,0,0,0,0,POINT (436131.824 585004.093)
21497,98.00,3.0,3.0,2.0,1925.0,53.124655,18.008459,0.59,49.0,0.113,...,0,1,1,0,0,1,0,0,0,POINT (433669.963 584830.108)
21498,108.96,5.0,2.0,4.0,1889.0,53.131748,18.000648,1.08,28.0,0.281,...,0,1,1,0,0,1,0,0,0,POINT (433158.461 585626.155)
21499,50.12,2.0,1.0,1.0,1993.0,53.129657,18.003888,0.88,42.0,0.174,...,1,0,1,0,0,1,0,0,0,POINT (433371.928 585390.604)


### Pozbywanie się duplikatów geograficznych metoda 1. - agregacja

In [35]:
from DataPreprocessing import df_aggregating


first_match_columns = ['floorCount', 'buildYear', 'centreDistance', 'poiCount',
        'schoolDistance', 'clinicDistance', 'postOfficeDistance', 'kindergartenDistance',
       'restaurantDistance', 'collegeDistance', 'pharmacyDistance', 'hasElevator',
        'hasSecurity', 'type_apartmentBuilding', 'type_blockOfFlats', 'type_tenement',
        'ownership_condominium', 'ownership_cooperative', 'buildingMaterial_brick',
       'buildingMaterial_concreteSlab', 'condition_low', 'condition_premium']

mean_match_columns = ['squareMeters', 'rooms', 'floor', 'hasParkingSpace', 'hasBalcony', 'hasStorageRoom', 'price', 'price_per_m2']

gdf_agg = df_aggregating(gdf, ['latitude', 'longitude'], {**{column: 'first' for column in first_match_columns}, **{column: 'mean' for column in mean_match_columns}})

In [36]:
gdf_agg.duplicated(subset=['geometry']).sum()

np.int64(0)

In [37]:
def create_GWR_model(y, gdf: gpd.GeoDataFrame, columns_with_features: list):
    y = y.reshape(-1, 1)
    X = gdf[columns_with_features]
    u = gdf.geometry.x
    v = gdf.geometry.y
    coords = list(zip(u, v))
    gwr_selector = Sel_BW(coords, y, X)
    gwr_bw = gwr_selector.search()
    return GWR(coords, y, X, gwr_bw).fit()


gwr_results = create_GWR_model(y=gdf_agg['price'].values, gdf=gdf_agg, columns_with_features=['squareMeters'])
lin_model = LinearRegression().fit(gdf_agg[['squareMeters']], gdf_agg['price'])

In [38]:
print('R2 =', gwr_results.R2)
print('AIC =', gwr_results.aic)
print('AICc =', gwr_results.aicc)
print('Linear model R2 = ', lin_model.score(gdf_agg[['squareMeters']], gdf_agg['price']))

R2 = 0.9038676841896828
AIC = 407654.12815526704
AICc = 408107.37526254397
Linear model R2 =  0.4621359326044805


In [39]:
def cross_validation_GWR(y, gdf, columns_with_features):
    y = y.reshape(-1, 1)
    X = gdf[columns_with_features]
    u = gdf.geometry.x
    v = gdf.geometry.y
    coords = list(zip(u, v))
    X_combined = np.hstack([X, coords])

    gwr_wrapper = SklearnGWR()

    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    scores = cross_val_score(gwr_wrapper, X_combined, y, cv=kf, scoring='r2')
    return scores


scores = cross_validation_GWR(y=gdf_agg['price'].values, gdf=gdf_agg, columns_with_features=['squareMeters'])
print(f"Średnie R^2: {scores.mean()}")

Znaleziono optymalne pasmo: 45.0
Znaleziono optymalne pasmo: 45.0
Znaleziono optymalne pasmo: 45.0
Znaleziono optymalne pasmo: 45.0
Znaleziono optymalne pasmo: 45.0
Średnie R^2: 0.8584112394635243


In [40]:
gdf_jiter = smart_jitter(gdf, scale=0.5)

print(f"Liczba wierszy: {len(gdf_jiter)}")
print(f"Liczba unikalnych XY: {len(gdf_jiter.drop_duplicates(subset=['x', 'y']))}")

Iteracja 1: Znaleziono 7050 punktów w kolizji. Nakładam jitter...
Wszystkie lokalizacje są unikalne po 1 iteracjach.
Liczba wierszy: 20474
Liczba unikalnych XY: 20474


In [63]:
y = gdf_jiter['price'].values.reshape((-1, 1))
y.shape

(20474, 1)

In [ ]:
gdf_jiter.columns

In [84]:
X = gdf_jiter[['squareMeters', 'buildYear']]
X.shape

(20474, 2)

In [71]:
u = gdf_jiter.geometry.x
v = gdf_jiter.geometry.y
coords = list(zip(u,v))

In [85]:
gwr_selector = Sel_BW(coords, y, X)
gwr_bw = gwr_selector.search()

print(gwr_bw)

51.0


In [ ]:
gwr_results = GWR(coords, y, X, gwr_bw).fit()
gwr_results.summary()

lin_model = LinearRegression()
lin_model.fit(X, y)